# Documentación del código de ClinicLog

ClinicLog es un sistema de consola desarrollado en Python para registrar pacientes,
asociar tratamientos y llevar un seguimiento básico de cada paciente.
:D

## Importaciones

Este bloque importa las herramientas necesarias para el funcionamiento del programa:

- `dataclass`: permite crear clases de datos de forma más corta.
- `datetime`: permite obtener la fecha y hora actual para los registros de seguimiento.
- `List` y `Optional`: indican los tipos de datos utilizados en listas y campos opcionales.
- `re`: permite validar que los nombres solo contengan letras y espacios.

In [1]:
from dataclasses import dataclass
from datetime import datetime
from typing import List, Optional
import re

## Función: limpiar_pantalla

La función `limpiar_pantalla` imprime líneas vacías para separar visualmente cada pantalla
del sistema en la consola. Esto mejora la presentación del menú y de los resultados.

In [2]:
def limpiar_pantalla():
    print("\n" * 2)

## Función: mostrar_encabezado

La función `mostrar_encabezado` recibe un título y lo muestra centrado entre líneas.
Se utiliza para identificar las diferentes secciones del sistema, como el menú principal,
registro de pacientes, tratamientos y seguimiento.

In [3]:
def mostrar_encabezado(titulo: str):
    linea = "=" * 40
    print(linea)
    print(titulo.center(40))
    print(linea)

## Función: leer_cadena

La función `leer_cadena` solicita un texto al usuario y elimina espacios al inicio y al final.

Recibe dos parámetros:

- `mensaje`: texto que se mostrará al usuario.
- `obligatorio`: indica si el campo puede quedar vacío.

Si el campo es obligatorio y el usuario no escribe ningún valor, la función muestra un error
y vuelve a solicitar el dato.

In [4]:
def leer_cadena(mensaje: str, obligatorio: bool = True) -> str:
    while True:
        valor = input(mensaje).strip()
        if valor or not obligatorio:
            return valor
        print("Error: este campo es obligatorio.")

## Función: leer_entero_rango

La función `leer_entero_rango` solicita un número entero y verifica que se encuentre
dentro de un rango establecido.

Recibe:

- `mensaje`: texto que se mostrará al usuario.
- `minimo`: valor mínimo permitido.
- `maximo`: valor máximo permitido.

Utiliza `try` y `except` para evitar que el programa se detenga cuando el usuario
ingresa letras, símbolos o valores que no pueden convertirse a número entero.

In [5]:
def leer_entero_rango(mensaje: str, minimo: int, maximo: int) -> int:
    while True:
        texto = input(mensaje).strip()
        try:
            numero = int(texto)
            if minimo <= numero <= maximo:
                return numero
            print(f"Error: ingrese un número entre {minimo} y {maximo}.")
        except ValueError:
            print("Error: ingrese un número entero válido.")

## Función: leer_decimal_positivo

La función `leer_decimal_positivo` solicita un número decimal y valida que no sea negativo.
Se utiliza para registrar el costo de un tratamiento.

Si el usuario ingresa un valor inválido o negativo, la función muestra un mensaje de error
y vuelve a solicitar el dato.

In [6]:
def leer_decimal_positivo(mensaje: str) -> float:
    while True:
        texto = input(mensaje).strip()
        try:
            numero = float(texto)
            if numero >= 0:
                return numero
            print("Error: el valor no puede ser negativo.")
        except ValueError:
            print("Error: ingrese un número decimal válido.")

## Función: confirmar

La función `confirmar` muestra una pregunta que solo se responde con `s`, `si`, `sí`
para confirmar una acción.

Retorna `True` cuando la respuesta es afirmativa y `False` para cualquier otra respuesta.
En ClinicLog se utiliza principalmente cuando se detecta un posible paciente duplicado.

In [7]:
def confirmar(mensaje: str) -> bool:
    resp = input(f"{mensaje} (s/n): ").strip().lower()
    return resp in ("s", "si", "sí")

## Función: generar_id

La función `generar_id` crea un identificador numérico usando la fecha y hora actual.
Este identificador se asigna a los objetos `Paciente`, `Tratamiento` y
`RegistroSeguimiento` para diferenciarlos dentro del sistema.

In [8]:
def generar_id() -> int:
    return int(datetime.now().timestamp() * 1000) % 1000000

## Función: validar_nombre

La función `validar_nombre` comprueba que un nombre tenga entre 2 y 50 caracteres
y que solo contenga letras, espacios, vocales con tilde y la letra ñ.

Se usa para validar los nombres y apellidos antes de registrar un paciente.

In [9]:
def validar_nombre(nombre: str) -> bool:
    if len(nombre) < 2 or len(nombre) > 50:
        return False
    return bool(re.match(r'^[A-Za-zÁÉÍÓÚáéíóúÑñ\s]+$', nombre))

## Clase: Paciente

La clase `Paciente` representa a una persona registrada en ClinicLog.

Sus atributos son:

- `id`: identificador único.
- `nombre`: nombre del paciente.
- `apellido`: apellido del paciente.
- `edad`: edad del paciente.
- `condicion_medica`: información opcional sobre su condición médica.

La función `nombre_completo` une el nombre y el apellido para facilitar su presentación.

In [10]:
@dataclass
class Paciente:
    id: int
    nombre: str
    apellido: str
    edad: int
    condicion_medica: Optional[str] = None

    def nombre_completo(self) -> str:
        return f"{self.nombre} {self.apellido}"

## Clase: Tratamiento

La clase `Tratamiento` representa un tratamiento asociado a un paciente.

Sus atributos son:

- `id`: identificador único del tratamiento.
- `id_paciente`: identificador del paciente al que pertenece.
- `nombre`: nombre del medicamento o tratamiento.
- `dosis`: cantidad indicada para el tratamiento.
- `frecuencia_horas`: tiempo en horas entre cada toma.
- `costo`: costo registrado para el tratamiento.

In [11]:
@dataclass
class Tratamiento:
    id: int
    id_paciente: int
    nombre: str
    dosis: str
    frecuencia_horas: int
    costo: float

## Clase: RegistroSeguimiento

La clase `RegistroSeguimiento` representa una toma, visita o anotación relacionada
con el seguimiento de un paciente.

Guarda el paciente, el tratamiento asociado, la fecha y una observación ingresada
por el usuario.

In [12]:
@dataclass
class RegistroSeguimiento:
    id: int
    id_paciente: int
    id_tratamiento: int
    fecha: datetime
    observaciones: str

## Clase: GestorPacientes

La clase `GestorPacientes` administra la lista de pacientes durante la ejecución
del programa.

Sus métodos permiten:

- Registrar un paciente.
- Obtener una lista ordenada de pacientes.
- Verificar si ya existe un paciente con el mismo nombre y apellido.

In [13]:
class GestorPacientes:
    def __init__(self):
        self.pacientes: List[Paciente] = []

    def registrar(self, paciente: Paciente):
        self.pacientes.append(paciente)

    def listar(self) -> List[Paciente]:
        return sorted(self.pacientes, key=lambda p: (p.nombre, p.apellido))

    def existe(self, nombre: str, apellido: str) -> bool:
        return any(
            p.nombre.lower() == nombre.lower() and p.apellido.lower() == apellido.lower()
            for p in self.pacientes
        )

## Clase: GestorTratamientos

La clase `GestorTratamientos` administra los tratamientos registrados.

Permite agregar tratamientos y obtener únicamente aquellos que pertenecen
a un paciente determinado mediante su identificador.

In [14]:
class GestorTratamientos:
    def __init__(self):
        self.tratamientos: List[Tratamiento] = []

    def registrar(self, tratamiento: Tratamiento):
        self.tratamientos.append(tratamiento)

    def por_paciente(self, id_paciente: int) -> List[Tratamiento]:
        return [t for t in self.tratamientos if t.id_paciente == id_paciente]

## Clase: GestorSeguimiento

La clase `GestorSeguimiento` administra los registros de seguimiento.

Permite:

- Agregar registros nuevos.
- Consultar todos los registros de un paciente.
- Consultar los registros de un tratamiento específico.

In [15]:
class GestorSeguimiento:
    def __init__(self):
        self.registros: List[RegistroSeguimiento] = []

    def agregar(self, registro: RegistroSeguimiento):
        self.registros.append(registro)

    def historial_paciente(self, id_paciente: int) -> List[RegistroSeguimiento]:
        return [r for r in self.registros if r.id_paciente == id_paciente]

    def historial_tratamiento(self, id_paciente: int, id_tratamiento: int) -> List[RegistroSeguimiento]:
        return [
            r for r in self.registros
            if r.id_paciente == id_paciente and r.id_tratamiento == id_tratamiento
        ]

## Función: inicializar_datos

La función `inicializar_datos` registra información de prueba al iniciar el sistema.

Crea dos pacientes, un tratamiento para cada uno y un registro inicial de seguimiento.
Los datos precargados permiten demostrar las funcionalidades sin tener que ingresar
toda la información manualmente antes de probar el programa.

In [16]:
def inicializar_datos(gestor_pacientes: GestorPacientes,
                      gestor_tratamientos: GestorTratamientos,
                      gestor_seguimiento: GestorSeguimiento) -> None:
    p1 = Paciente(generar_id(), "Germán", "Pérez", 35, "Hipertensión")
    p2 = Paciente(generar_id(), "Ana", "Gómez", 28, "Diabetes tipo 2")

    gestor_pacientes.registrar(p1)
    gestor_pacientes.registrar(p2)

    t1 = Tratamiento(generar_id(), p1.id, "Losartán", "50mg", 24, 150.0)
    t2 = Tratamiento(generar_id(), p2.id, "Metformina", "850mg", 12, 80.0)

    gestor_tratamientos.registrar(t1)
    gestor_tratamientos.registrar(t2)

    ahora = datetime.now()
    gestor_seguimiento.agregar(
        RegistroSeguimiento(generar_id(), p1.id, t1.id, ahora, "Toma de prueba")
    )
    gestor_seguimiento.agregar(
        RegistroSeguimiento(generar_id(), p2.id, t2.id, ahora, "Control inicial")
    )

## Función: registrar_paciente_menu

Esta función implementa la primera opción del menú principal: registrar un paciente.

El proceso es el siguiente:

1. Solicita nombre y apellido.
2. Valida que ambos contengan únicamente letras y tengan la longitud permitida.
3. Verifica si ya existe un paciente con el mismo nombre y apellido.
4. Solicita y valida la edad.
5. Solicita una condición médica opcional.
6. Crea un objeto `Paciente` y lo registra en `GestorPacientes`.

La función evita registros incompletos y controla los posibles duplicados.

In [17]:
def registrar_paciente_menu(gestor_pacientes: GestorPacientes) -> None:
    mostrar_encabezado("REGISTRAR PACIENTE")

    while True:
        nombre = leer_cadena("Nombre: ")
        if validar_nombre(nombre):
            break
        print("Error: el nombre solo puede contener letras (2-50 caracteres).")

    while True:
        apellido = leer_cadena("Apellido: ")
        if validar_nombre(apellido):
            break
        print("Error: el apellido solo puede contener letras (2-50 caracteres).")

    if gestor_pacientes.existe(nombre, apellido):
        print("Advertencia: ya existe un paciente con este nombre y apellido.")
        if not confirmar("¿Desea continuar y registrar otro?"):
            print("Operación cancelada.")
            input("Presione Enter para continuar...")
            return

    edad = leer_entero_rango("Edad: ", 0, 120)

    condicion = leer_cadena(
        "Condición médica (opcional, Enter para saltar): ",
        obligatorio=False
    )
    if condicion == "":
        condicion = None

    paciente = Paciente(generar_id(), nombre, apellido, edad, condicion)
    gestor_pacientes.registrar(paciente)

    print(f"Paciente {paciente.nombre_completo()} registrado.")
    input("Presione Enter para continuar...")

## Función: registrar_tratamiento_menu

Esta función implementa la segunda opción del menú principal: registrar un tratamiento.

Primero obtiene y muestra los pacientes existentes. Luego permite seleccionar uno y solicita:

- Nombre del tratamiento.
- Dosis.
- Frecuencia en horas.
- Costo.

La frecuencia y el costo son validados antes de crear el objeto `Tratamiento`.
Finalmente, el tratamiento queda asociado al paciente seleccionado mediante el atributo
`id_paciente`.

In [18]:
def registrar_tratamiento_menu(gestor_pacientes: GestorPacientes,
                               gestor_tratamientos: GestorTratamientos) -> None:
    mostrar_encabezado("REGISTRAR TRATAMIENTO")

    pacientes = gestor_pacientes.listar()
    if not pacientes:
        print("No hay pacientes registrados.")
        input("Presione Enter para continuar...")
        return

    for i, p in enumerate(pacientes, 1):
        print(f"{i}. {p.nombre_completo()} ({p.edad} años)")

    indice = leer_entero_rango("Seleccione paciente (número): ", 1, len(pacientes))
    paciente = pacientes[indice - 1]

    nombre = leer_cadena("Nombre del tratamiento: ")
    dosis = leer_cadena("Dosis: ")
    frecuencia = leer_entero_rango("Frecuencia (horas entre tomas): ", 1, 365)
    costo = leer_decimal_positivo("Costo: ")

    tratamiento = Tratamiento(
        generar_id(),
        paciente.id,
        nombre,
        dosis,
        frecuencia,
        costo
    )

    gestor_tratamientos.registrar(tratamiento)

    print(f"Tratamiento '{nombre}' registrado para {paciente.nombre_completo()}.")
    input("Presione Enter para continuar...")

## Función: ver_pacientes_menu

Esta función implementa la tercera opción del menú: ver pacientes.

Obtiene la lista ordenada de pacientes y, para cada uno, muestra:

- Nombre completo.
- Edad.
- Condición médica, si fue registrada.
- Tratamientos asociados, si existen.

No modifica información; únicamente consulta y presenta los datos almacenados.

In [19]:
def ver_pacientes_menu(gestor_pacientes: GestorPacientes,
                       gestor_tratamientos: GestorTratamientos) -> None:
    mostrar_encabezado("LISTA DE PACIENTES")

    pacientes = gestor_pacientes.listar()
    if not pacientes:
        print("No hay pacientes registrados.")
        input("Presione Enter para continuar...")
        return

    for i, p in enumerate(pacientes, 1):
        print(f"{i}. {p.nombre_completo()} - {p.edad} años")

        if p.condicion_medica:
            print(f"   Condición: {p.condicion_medica}")

        tratamientos = gestor_tratamientos.por_paciente(p.id)
        if tratamientos:
            print("   Tratamientos:")
            for t in tratamientos:
                print(
                    f"      - {t.nombre} ({t.dosis}) cada "
                    f"{t.frecuencia_horas}h, costo {t.costo}"
                )
        else:
            print("   Sin tratamientos")

        print()

    input("Presione Enter para continuar...")

## Función: seguimiento_paciente_menu

Esta función implementa la cuarta opción del menú: seguimiento de paciente.

Primero permite seleccionar un paciente. Luego muestra un submenú con cuatro opciones:

1. Ver el historial completo del paciente.
2. Ver el historial de un tratamiento específico.
3. Registrar una nueva toma o visita.
4. Volver al menú principal.

Cuando se registra una toma o visita, el sistema solicita observaciones, crea un objeto
`RegistroSeguimiento` con la fecha actual y lo almacena en `GestorSeguimiento`.

In [20]:
def seguimiento_paciente_menu(gestor_pacientes: GestorPacientes,
                              gestor_tratamientos: GestorTratamientos,
                              gestor_seguimiento: GestorSeguimiento) -> None:
    mostrar_encabezado("SEGUIMIENTO DE PACIENTE")

    pacientes = gestor_pacientes.listar()
    if not pacientes:
        print("No hay pacientes registrados.")
        input("Presione Enter para continuar...")
        return

    for i, p in enumerate(pacientes, 1):
        print(f"{i}. {p.nombre_completo()}")

    indice = leer_entero_rango("Seleccione paciente (número): ", 1, len(pacientes))
    paciente = pacientes[indice - 1]
    tratamientos = gestor_tratamientos.por_paciente(paciente.id)

    while True:
        mostrar_encabezado(f"SEGUIMIENTO: {paciente.nombre_completo()}")
        print("1. Ver historial completo")
        print("2. Ver historial de un tratamiento")
        print("3. Registrar nueva toma/visita")
        print("4. Volver")

        opcion = leer_entero_rango("Opción: ", 1, 4)

        if opcion == 1:
            historial = gestor_seguimiento.historial_paciente(paciente.id)

            if not historial:
                print("No hay registros para este paciente.")
            else:
                for r in historial:
                    print(f"- {r.fecha.strftime('%d/%m/%Y %H:%M')} | {r.observaciones}")

            input("Presione Enter para continuar...")

        elif opcion == 2:
            if not tratamientos:
                print("Este paciente no tiene tratamientos.")
                input("Presione Enter para continuar...")
                continue

            for i, t in enumerate(tratamientos, 1):
                print(f"{i}. {t.nombre}")

            sel = leer_entero_rango("Seleccione tratamiento: ", 1, len(tratamientos))
            tratamiento = tratamientos[sel - 1]

            historial = gestor_seguimiento.historial_tratamiento(
                paciente.id,
                tratamiento.id
            )

            if not historial:
                print("No hay registros para este tratamiento.")
            else:
                for r in historial:
                    print(f"- {r.fecha.strftime('%d/%m/%Y %H:%M')} | {r.observaciones}")

            input("Presione Enter para continuar...")

        elif opcion == 3:
            if tratamientos:
                print("Tratamientos del paciente:")

                for i, t in enumerate(tratamientos, 1):
                    print(f"{i}. {t.nombre}")

                sel = leer_entero_rango(
                    "Seleccione tratamiento: ",
                    1,
                    len(tratamientos)
                )

                tratamiento = tratamientos[sel - 1]
                id_tratamiento = tratamiento.id
            else:
                id_tratamiento = 0

            observaciones = leer_cadena("Observaciones: ")

            registro = RegistroSeguimiento(
                generar_id(),
                paciente.id,
                id_tratamiento,
                datetime.now(),
                observaciones
            )

            gestor_seguimiento.agregar(registro)

            print("Registro agregado.")
            input("Presione Enter para continuar...")

        elif opcion == 4:
            break

## Función: menu_principal

La función `menu_principal` muestra las cinco opciones que el usuario puede seleccionar
durante la ejecución del sistema:

1. Registrar paciente.
2. Registrar tratamiento.
3. Ver pacientes.
4. Seguimiento de paciente.
5. Salir.

Esta función solamente muestra las opciones; la selección y ejecución se realizan en `main`.

In [21]:
def menu_principal():
    mostrar_encabezado("CLINICLOG - MENÚ PRINCIPAL")
    print("1. Registrar paciente")
    print("2. Registrar tratamiento")
    print("3. Ver pacientes")
    print("4. Seguimiento de paciente")
    print("5. Salir")

## Función: main

La función `main` controla el funcionamiento general de ClinicLog.

Al iniciar:

1. Crea los tres gestores del sistema.
2. Carga los datos de prueba.
3. Muestra el menú principal dentro de un ciclo repetitivo.
4. Lee y valida la opción seleccionada.
5. Llama a la función correspondiente.
6. Finaliza el programa cuando el usuario selecciona la opción 5.

Esta función es el punto principal de ejecución del algoritmo.

In [22]:
def main():
    gestor_pacientes = GestorPacientes()
    gestor_tratamientos = GestorTratamientos()
    gestor_seguimiento = GestorSeguimiento()

    inicializar_datos(gestor_pacientes, gestor_tratamientos, gestor_seguimiento)

    while True:
        limpiar_pantalla()
        menu_principal()

        opcion = leer_entero_rango("Seleccione una opción (1-5): ", 1, 5)

        if opcion == 1:
            registrar_paciente_menu(gestor_pacientes)

        elif opcion == 2:
            registrar_tratamiento_menu(
                gestor_pacientes,
                gestor_tratamientos
            )

        elif opcion == 3:
            ver_pacientes_menu(
                gestor_pacientes,
                gestor_tratamientos
            )

        elif opcion == 4:
            seguimiento_paciente_menu(
                gestor_pacientes,
                gestor_tratamientos,
                gestor_seguimiento
            )

        elif opcion == 5:
            mostrar_encabezado("GRACIAS POR USAR CLINICLOG")
            break

# Pseudocódigo de ClinicLog

## Algoritmo principal

INICIO

    Crear gestor de pacientes
    Crear gestor de tratamientos
    Crear gestor de seguimiento

    Inicializar datos de prueba

    REPETIR
        Limpiar pantalla
        Mostrar menú principal

        Leer opción entre 1 y 5

        SEGÚN opción HACER

            CASO 1:
                Registrar paciente

            CASO 2:
                Registrar tratamiento

            CASO 3:
                Mostrar pacientes

            CASO 4:
                Mostrar menú de seguimiento de paciente

            CASO 5:
                Mostrar mensaje de despedida
                Terminar programa

        FIN SEGÚN

    HASTA QUE la opción sea 5

FIN


## Procedimiento Registrar paciente

INICIO

    Mostrar encabezado de registro de paciente

    REPETIR
        Solicitar nombre
        Validar que tenga entre 2 y 50 caracteres
        Validar que solo tenga letras y espacios
    HASTA QUE el nombre sea válido

    REPETIR
        Solicitar apellido
        Validar que tenga entre 2 y 50 caracteres
        Validar que solo tenga letras y espacios
    HASTA QUE el apellido sea válido

    SI existe un paciente con el mismo nombre y apellido ENTONCES
        Mostrar advertencia
        Preguntar si desea continuar

        SI la respuesta es negativa ENTONCES
            Mostrar mensaje de operación cancelada
            Terminar procedimiento
        FIN SI
    FIN SI

    Solicitar edad
    Validar que sea un número entero entre 0 y 120

    Solicitar condición médica opcional

    Generar identificador del paciente
    Crear objeto paciente
    Registrar paciente en la lista de pacientes

    Mostrar mensaje de registro exitoso

FIN


## Procedimiento Registrar tratamiento

INICIO

    Mostrar encabezado de registro de tratamiento

    Obtener lista de pacientes

    SI no hay pacientes registrados ENTONCES
        Mostrar mensaje
        Terminar procedimiento
    FIN SI

    Mostrar pacientes numerados
    Solicitar número de paciente
    Validar que el número esté dentro del rango permitido

    Obtener paciente seleccionado

    Solicitar nombre del tratamiento
    Solicitar dosis

    Solicitar frecuencia de toma
    Validar que sea un número entero entre 1 y 365

    Solicitar costo
    Validar que sea un número decimal mayor o igual a cero

    Generar identificador del tratamiento
    Crear objeto tratamiento asociado al paciente seleccionado
    Registrar tratamiento en la lista de tratamientos

    Mostrar mensaje de registro exitoso

FIN


## Procedimiento Ver pacientes

INICIO

    Mostrar encabezado de lista de pacientes

    Obtener lista de pacientes ordenada por nombre y apellido

    SI no hay pacientes registrados ENTONCES
        Mostrar mensaje
        Terminar procedimiento
    FIN SI

    PARA cada paciente de la lista HACER

        Mostrar nombre completo y edad

        SI el paciente tiene condición médica ENTONCES
            Mostrar condición médica
        FIN SI

        Obtener tratamientos del paciente

        SI el paciente tiene tratamientos ENTONCES
            Mostrar cada tratamiento con dosis, frecuencia y costo
        SINO
            Mostrar mensaje "Sin tratamientos"
        FIN SI

    FIN PARA

FIN


## Procedimiento Seguimiento de paciente

INICIO

    Mostrar encabezado de seguimiento

    Obtener lista de pacientes

    SI no hay pacientes registrados ENTONCES
        Mostrar mensaje
        Terminar procedimiento
    FIN SI

    Mostrar pacientes numerados
    Solicitar número de paciente
    Validar selección

    Obtener paciente seleccionado
    Obtener tratamientos asociados al paciente

    REPETIR

        Mostrar menú de seguimiento:
            1. Ver historial completo
            2. Ver historial de un tratamiento
            3. Registrar nueva toma o visita
            4. Volver

        Leer opción entre 1 y 4

        SEGÚN opción HACER

            CASO 1:
                Obtener historial del paciente

                SI no existen registros ENTONCES
                    Mostrar mensaje
                SINO
                    Mostrar fecha y observaciones de cada registro
                FIN SI

            CASO 2:
                SI el paciente no tiene tratamientos ENTONCES
                    Mostrar mensaje
                SINO
                    Mostrar tratamientos
                    Solicitar tratamiento seleccionado
                    Obtener historial del tratamiento

                    SI no existen registros ENTONCES
                        Mostrar mensaje
                    SINO
                        Mostrar fecha y observaciones de cada registro
                    FIN SI
                FIN SI

            CASO 3:
                SI el paciente tiene tratamientos ENTONCES
                    Mostrar tratamientos
                    Solicitar tratamiento seleccionado
                    Guardar identificador del tratamiento
                SINO
                    Usar identificador de tratamiento igual a 0
                FIN SI

                Solicitar observaciones

                Generar identificador de seguimiento
                Obtener fecha y hora actual
                Crear registro de seguimiento
                Agregar registro al historial

                Mostrar mensaje de registro agregado

            CASO 4:
                Volver al menú principal

        FIN SEGÚN

    HASTA QUE la opción sea 4

FIN